# Chuẩn bị dataset YOLO11 detection 3 lớp

Nguồn là ZIP Roboflow 4 lớp `bike`, `helmet`, `no-helmet`, `number-plate`. Notebook xóa lớp `number-plate`, chuyển nhãn polygon sang bbox YOLO, cắt box vượt mép ảnh, bỏ box diện tích 0 và nhãn trùng hệt nhau. Nó kiểm tra ảnh–nhãn, ảnh hỏng và ảnh trùng chính xác giữa các split, rồi xuất ZIP 3 lớp và báo cáo chất lượng.

Chỉ **bản giải nén tạm trong `/content`** được sửa. ZIP 4 lớp và ZIP 3 lớp cũ trên Drive được giữ nguyên. ZIP mới có hậu tố `_clean.zip`. Các lớp 0–2 được giữ nguyên. Để giảm rò rỉ dữ liệu, các biến thể Roboflow cùng ảnh nguồn sẽ chỉ nằm ở một split (ưu tiên test, rồi valid, rồi train). Notebook không tự tạo dữ liệu người đi bộ hay cân bằng lớp bằng ảnh lặp.

## 1. Cấu hình đường dẫn

In [ ]:
from pathlib import Path

SOURCE_ZIP = Path(
    "/content/drive/MyDrive/Colab/"
    "Helmet and Number Plate Detection for Motorbike Safety.v1-datasetdefault.yolov11.zip"
)
OUTPUT_ZIP = Path("/content/drive/MyDrive/Colab/helmet_dataset_3classes_clean.zip")
WORK_DIR = Path("/content/helmet_dataset_3classes_clean_build")

EXPECTED_SOURCE_NAMES = ["bike", "helmet", "no-helmet", "number-plate"]
TARGET_NAMES = EXPECTED_SOURCE_NAMES[:3]
REMOVE_CLASS_ID = 3
SPLITS = {"train": "train", "val": "valid", "test": "test"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
REMOVE_CROSS_SPLIT_ORIGIN_OVERLAP = True
KEEP_ONE_EVAL_IMAGE_PER_ORIGIN = True

print("ZIP 4 lớp:", SOURCE_ZIP)
print("ZIP 3 lớp mới:", OUTPUT_ZIP)

## 2. Gắn Google Drive và kiểm tra đầu vào

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"Không tìm thấy ZIP nguồn: {SOURCE_ZIP}")
if OUTPUT_ZIP.exists():
    raise FileExistsError(
        f"ZIP đầu ra đã tồn tại: {OUTPUT_ZIP}. Đổi OUTPUT_ZIP nếu muốn tạo phiên bản mới."
    )
if SOURCE_ZIP.resolve() == OUTPUT_ZIP.resolve():
    raise ValueError("ZIP nguồn và ZIP đầu ra phải khác nhau")

print(f"Nguồn: {SOURCE_ZIP.stat().st_size / 1024**3:.2f} GiB")

## 3. Giải nén tạm và kiểm tra cấu trúc dataset

In [ ]:
import shutil
import yaml
from zipfile import ZipFile

if WORK_DIR != Path("/content/helmet_dataset_3classes_clean_build"):
    raise ValueError("WORK_DIR phải là thư mục tạm đã định trong /content")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

with ZipFile(SOURCE_ZIP) as archive:
    entries = archive.infolist()
    uncompressed_bytes = sum(info.file_size for info in entries)
    needed = uncompressed_bytes + SOURCE_ZIP.stat().st_size + 1024**3
    free = shutil.disk_usage("/content").free
    if free < needed:
        raise OSError(
            f"Không đủ dung lượng trống /content: cần khoảng {needed/1024**3:.1f} GiB, "
            f"còn {free/1024**3:.1f} GiB"
        )
    for info in entries:
        destination = (WORK_DIR / info.filename).resolve()
        if not destination.is_relative_to(WORK_DIR.resolve()):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {info.filename}")
    archive.extractall(WORK_DIR)

yaml_files = [p for p in WORK_DIR.rglob("data.yaml") if "__MACOSX" not in p.parts]
if len(yaml_files) != 1:
    raise ValueError(f"Cần đúng một data.yaml sau giải nén, có: {yaml_files}")
DATASET_ROOT = yaml_files[0].parent
with yaml_files[0].open(encoding="utf-8") as f:
    SOURCE_CFG = yaml.safe_load(f)

source_names = SOURCE_CFG.get("names")
if isinstance(source_names, dict):
    source_names = [source_names[i] if i in source_names else source_names[str(i)]
                    for i in range(len(source_names))]
if source_names != EXPECTED_SOURCE_NAMES or int(SOURCE_CFG.get("nc", -1)) != 4:
    raise ValueError(
        f"Thứ tự lớp nguồn không như dự kiến: nc={SOURCE_CFG.get('nc')}, names={source_names}"
    )

SPLIT_FILES = {}
for split, folder in SPLITS.items():
    image_dir = DATASET_ROOT / folder / "images"
    label_dir = DATASET_ROOT / folder / "labels"
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Thiếu images hoặc labels của {split}: {image_dir}, {label_dir}")
    images = [p for p in image_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    labels = list(label_dir.rglob("*.txt"))
    image_map = {str(p.relative_to(image_dir).with_suffix("")): p for p in images}
    label_map = {str(p.relative_to(label_dir).with_suffix("")): p for p in labels}
    if len(image_map) != len(images):
        raise ValueError(f"Có nhiều ảnh cùng tên gốc nhưng khác đuôi trong split {split}")
    missing = sorted(set(image_map) - set(label_map))
    orphan = sorted(set(label_map) - set(image_map))
    if missing or orphan:
        raise ValueError(f"{split}: thiếu nhãn {missing[:5]}, nhãn không có ảnh {orphan[:5]}")
    if not images:
        raise ValueError(f"Split {split} không có ảnh")
    SPLIT_FILES[split] = {"images": image_map, "labels": label_map}
    print(f"{split:5s}: {len(images)} ảnh và {len(labels)} nhãn khớp nhau")

print("Lớp nguồn:", dict(enumerate(source_names)))

## 3b. Loại biến thể cùng ảnh nguồn khỏi nhiều split

Roboflow thêm hậu tố `.rf.<hash>` vào tên ảnh. Các ảnh có cùng phần tên trước `.rf.` được coi là cùng ảnh nguồn để tránh train và test nhìn thấy các phiên bản gần giống nhau. Bản mặc định giữ một biến thể cho mỗi ảnh nguồn ở valid/test, sau đó ưu tiên **test → valid → train** khi loại ảnh trùng nguồn giữa split. Số ảnh loại được ghi vào báo cáo. Đây là quy tắc bảo thủ theo tên ảnh; nếu tên nguồn trùng nhưng ảnh thật khác nhau, có thể đặt hai tùy chọn ở bước 1 thành `False` để giữ split gốc.

In [ ]:
from collections import Counter

def source_stem(stem):
    return stem.rsplit(".rf.", 1)[0]

REMOVED_SPLIT_OVERLAP = Counter()
REMOVED_EVAL_VARIANTS = Counter()
REMOVED_SPLIT_EXAMPLES = []
original_origins = {
    split: {source_stem(stem) for stem in files["images"]}
    for split, files in SPLIT_FILES.items()
}
overlap_before = {
    f"{a}-{b}": len(original_origins[a] & original_origins[b])
    for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
}
print("Tên ảnh nguồn chung trước xử lý:", overlap_before)

if KEEP_ONE_EVAL_IMAGE_PER_ORIGIN:
    for split in ("test", "val"):
        files = SPLIT_FILES[split]
        seen_origins = set()
        for stem in sorted(list(files["images"])):
            origin = source_stem(stem)
            if origin in seen_origins:
                files["images"].pop(stem).unlink()  # chỉ xóa bản giải nén tạm
                files["labels"].pop(stem).unlink()
                REMOVED_EVAL_VARIANTS[split] += 1
            else:
                seen_origins.add(origin)

if REMOVE_CROSS_SPLIT_ORIGIN_OVERLAP:
    claimed_origins = set()
    for split in ("test", "val", "train"):
        files = SPLIT_FILES[split]
        keep_origins = set()
        for stem in list(files["images"]):
            origin = source_stem(stem)
            if origin in claimed_origins:
                image_path = files["images"].pop(stem)
                label_path = files["labels"].pop(stem)
                image_path.unlink()  # chỉ xóa bản giải nén tạm trong /content
                label_path.unlink()
                REMOVED_SPLIT_OVERLAP[split] += 1
                if len(REMOVED_SPLIT_EXAMPLES) < 20:
                    REMOVED_SPLIT_EXAMPLES.append(str(image_path))
            else:
                keep_origins.add(origin)
        claimed_origins.update(keep_origins)
        if not files["images"]:
            raise ValueError(f"Split {split} hết ảnh sau khi loại trùng nguồn")

print("Ảnh bỏ khỏi bản tạm để chống rò rỉ:", dict(REMOVED_SPLIT_OVERLAP))
print("Biến thể dư ở valid/test đã bỏ:", dict(REMOVED_EVAL_VARIANTS))
print("Ảnh còn lại:", {k: len(v["images"]) for k, v in SPLIT_FILES.items()})

## 4. Xóa `number-plate` và chuẩn hóa nhãn detection

Mỗi polygon được đổi thành bbox bao quanh các điểm. Bbox vượt mép ảnh được cắt vào `[0,1]`; box có diện tích 0 bị bỏ và được ghi vào báo cáo. Ảnh chỉ có `number-plate` sẽ có file nhãn rỗng để làm ảnh nền. Đây là biến đổi trên dữ liệu tạm, không sửa ZIP nguồn.

In [ ]:
import math
from collections import Counter

QUALITY_STATS = {}
CONVERTED_SAMPLES = []
DROPPED_EXAMPLES = []
CLIPPED_EXAMPLES = []

def clamp01(value):
    return max(0.0, min(1.0, value))

def to_clipped_box(x1, y1, x2, y2):
    clipped = any(v < 0 or v > 1 for v in (x1, y1, x2, y2))
    x1, y1, x2, y2 = map(clamp01, (x1, y1, x2, y2))
    if x2 <= x1 or y2 <= y1:
        return None, clipped
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1), clipped

for split, files in SPLIT_FILES.items():
    counters = Counter()
    class_counts = Counter()
    for stem, label_path in files["labels"].items():
        new_lines = []
        seen = set()
        for line_no, line in enumerate(label_path.read_text(encoding="utf-8-sig").splitlines(), 1):
            parts = line.split()
            if not parts:
                continue
            counters["source_annotations"] += 1
            try:
                class_value = float(parts[0])
                class_id = int(class_value)
            except (ValueError, OverflowError) as exc:
                raise ValueError(f"ID lớp không phải số tại {label_path}:{line_no}") from exc
            if not math.isfinite(class_value) or class_value != class_id or class_id not in (0, 1, 2, 3):
                raise ValueError(f"ID lớp không hợp lệ tại {label_path}:{line_no}: {parts[0]}")
            if class_id == REMOVE_CLASS_ID:
                counters["removed_number_plate"] += 1
                continue

            try:
                values = [float(v) for v in parts[1:]]
            except ValueError as exc:
                raise ValueError(f"Tọa độ không phải số tại {label_path}:{line_no}") from exc
            if not all(math.isfinite(v) and 0 <= v <= 1 for v in values):
                raise ValueError(f"Tọa độ ngoài [0,1] tại {label_path}:{line_no}")

            if len(parts) == 5:
                counters["source_bbox"] += 1
                xc, yc, w, h = values
                box, clipped = to_clipped_box(
                    xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2
                )
                if clipped:
                    counters["clipped_bbox"] += 1
                    if len(CLIPPED_EXAMPLES) < 20:
                        CLIPPED_EXAMPLES.append(f"{label_path}:{line_no}")
            elif len(parts) >= 7 and len(parts) % 2 == 1:
                counters["converted_polygon"] += 1
                xs, ys = values[0::2], values[1::2]
                box, _ = to_clipped_box(min(xs), min(ys), max(xs), max(ys))
                if len(CONVERTED_SAMPLES) < 8:
                    CONVERTED_SAMPLES.append((split, stem))
            else:
                raise ValueError(
                    f"Nhãn {label_path}:{line_no} có {len(parts)} giá trị; "
                    f"cần bbox 5 giá trị hoặc polygon 7+ số lẻ. Đầu dòng: {line[:100]!r}"
                )

            if box is None:
                counters["dropped_degenerate"] += 1
                if len(DROPPED_EXAMPLES) < 20:
                    DROPPED_EXAMPLES.append(f"{label_path}:{line_no}")
                continue
            out_line = str(class_id) + " " + " ".join(f"{v:.8f}" for v in box)
            if out_line in seen:
                counters["removed_exact_duplicates"] += 1
                continue
            seen.add(out_line)
            new_lines.append(out_line)
            class_counts[class_id] += 1

        if not new_lines:
            counters["empty_label_images"] += 1
        label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""), encoding="utf-8")

    if any(class_counts[i] == 0 for i in (0, 1, 2)):
        raise ValueError(f"Split {split} thiếu một trong ba lớp: {dict(class_counts)}")
    QUALITY_STATS[split] = {
        "images": len(files["images"]),
        "label_files": len(files["labels"]),
        "annotations_by_class": {TARGET_NAMES[i]: class_counts[i] for i in (0, 1, 2)},
        **dict(counters),
    }
    print(split, QUALITY_STATS[split])

print("Ví dụ box bị bỏ vì diện tích 0:", DROPPED_EXAMPLES[:5])

## 5. Kiểm tra ảnh, nhãn sau chuẩn hóa và rò rỉ split

Kiểm tra ảnh giải mã được, mọi dòng nhãn mới có đúng 5 giá trị và không có ảnh trùng **chính xác** giữa các split. Nếu đã bật bước 3b, các biến thể cùng tên nguồn ở nhiều split đã được loại khỏi bản tạm và số lượng được ghi vào báo cáo.

In [ ]:
import hashlib
from collections import defaultdict
from PIL import Image

seen_hashes = {}
cross_split_duplicates = []
source_stems = defaultdict(set)
dimensions = Counter()

for split, files in SPLIT_FILES.items():
    for stem, image_path in files["images"].items():
        with Image.open(image_path) as image:
            dimensions[image.size] += 1
            image.verify()
        h = hashlib.sha256()
        with image_path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(chunk)
        digest = h.hexdigest()
        if digest in seen_hashes and seen_hashes[digest][0] != split:
            cross_split_duplicates.append((seen_hashes[digest], (split, str(image_path))))
        else:
            seen_hashes[digest] = (split, str(image_path))
        source_stems[source_stem(stem)].add(split)

    for label_path in files["labels"].values():
        for line_no, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f"Nhãn sau chuẩn hóa chưa phải bbox: {label_path}:{line_no}")
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            if cls not in (0, 1, 2) or w <= 0 or h <= 0:
                raise ValueError(f"Nhãn sau chuẩn hóa không hợp lệ: {label_path}:{line_no}")
            if x-w/2 < -1e-7 or y-h/2 < -1e-7 or x+w/2 > 1+1e-7 or y+h/2 > 1+1e-7:
                raise ValueError(f"Bbox vượt mép sau chuẩn hóa: {label_path}:{line_no}")

if cross_split_duplicates:
    raise ValueError(f"Có ảnh trùng chính xác giữa các split: {cross_split_duplicates[:3]}")
SHARED_SOURCE_STEMS = sum(len(v) > 1 for v in source_stems.values())
if REMOVE_CROSS_SPLIT_ORIGIN_OVERLAP and SHARED_SOURCE_STEMS:
    raise RuntimeError("Vẫn còn ảnh cùng nguồn ở nhiều split")
print("Ảnh trùng chính xác giữa split:", len(cross_split_duplicates))
print("Tên nguồn giống nhau giữa split (cần xem xét thủ công):", SHARED_SOURCE_STEMS)
print("Kích thước ảnh phổ biến:", dimensions.most_common(5))

## 6. Ghi `data.yaml` và báo cáo kiểm tra

In [ ]:
import csv
import json
from datetime import datetime

new_cfg = {
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": TARGET_NAMES,
}
if SOURCE_CFG.get("roboflow"):
    new_cfg["roboflow"] = SOURCE_CFG["roboflow"]
with (DATASET_ROOT / "data.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(new_cfg, f, sort_keys=False, allow_unicode=True)

report_dir = DATASET_ROOT / "_quality_report"
report_dir.mkdir(exist_ok=True)
report = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_zip": str(SOURCE_ZIP),
    "output_zip": str(OUTPUT_ZIP),
    "source_classes": source_names,
    "target_classes": TARGET_NAMES,
    "split_stats": QUALITY_STATS,
    "exact_duplicate_images_across_splits": len(cross_split_duplicates),
    "source_stems_shared_across_splits": SHARED_SOURCE_STEMS,
    "origin_overlap_before": overlap_before,
    "removed_images_by_split_to_prevent_leakage": dict(REMOVED_SPLIT_OVERLAP),
    "removed_extra_eval_variants": dict(REMOVED_EVAL_VARIANTS),
    "sample_removed_images": REMOVED_SPLIT_EXAMPLES,
    "sample_dropped_degenerate_labels": DROPPED_EXAMPLES,
    "sample_clipped_bbox_labels": CLIPPED_EXAMPLES,
    "method": "Remove class 3; convert polygons to enclosing YOLO detection boxes; clip; deduplicate; prevent source-image overlap across splits",
}
(report_dir / "summary.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
with (report_dir / "class_counts.csv").open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["split", "class_id", "class_name", "annotations"])
    for split, stats in QUALITY_STATS.items():
        for class_id, name in enumerate(TARGET_NAMES):
            writer.writerow([split, class_id, name, stats["annotations_by_class"][name]])

print((DATASET_ROOT / "data.yaml").read_text(encoding="utf-8"))
print("Báo cáo:", report_dir)

## 7. Xem nhanh ảnh có nhãn polygon đã chuyển

Đây là bước kiểm tra trực quan. Hãy xem box mới có bao đúng xe/đầu người hay không trước khi train.

In [ ]:
from PIL import ImageDraw
from IPython.display import display

colors = {0: "#2E86DE", 1: "#27AE60", 2: "#E74C3C"}
for split, stem in CONVERTED_SAMPLES[:6]:
    image_path = SPLIT_FILES[split]["images"][stem]
    label_path = SPLIT_FILES[split]["labels"][stem]
    with Image.open(image_path) as src:
        original_w, original_h = src.size
        preview = src.convert("RGB")
    preview.thumbnail((700, 450))
    draw = ImageDraw.Draw(preview)
    for line in label_path.read_text(encoding="utf-8").splitlines():
        cls, x, y, w, h = map(float, line.split())
        x1, y1 = (x-w/2)*preview.width, (y-h/2)*preview.height
        x2, y2 = (x+w/2)*preview.width, (y+h/2)*preview.height
        color = colors[int(cls)]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1, max(0, y1-14)), TARGET_NAMES[int(cls)], fill=color)
    print(split, image_path.name, f"({original_w}x{original_h})")
    display(preview)

## 8. Đóng gói ZIP sạch và lưu lên Drive

Chỉ thực hiện sau khi các bước kiểm tra hoàn tất. ZIP được tạo và kiểm tra CRC ở `/content`, sau đó mới chép sang Drive. Notebook không ghi đè ZIP 3 lớp cũ.

In [ ]:
import os

if OUTPUT_ZIP.exists():
    raise FileExistsError(f"Đã có ZIP đầu ra: {OUTPUT_ZIP}. Hãy đổi tên đầu ra.")
local_base = Path("/content/helmet_dataset_3classes_clean")
local_zip = local_base.with_suffix(".zip")
if local_zip.exists():
    local_zip.unlink()  # chỉ xóa bản ZIP tạm ở /content từ lần chạy trước
shutil.make_archive(str(local_base), "zip", root_dir=str(DATASET_ROOT))

with ZipFile(local_zip) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP mới bị lỗi CRC ở: {bad_member}")
    names_in_zip = set(archive.namelist())
    if not {"data.yaml", "_quality_report/summary.json"}.issubset(names_in_zip):
        raise RuntimeError("ZIP mới thiếu data.yaml hoặc báo cáo chất lượng")

uploading = OUTPUT_ZIP.with_name(OUTPUT_ZIP.name + ".uploading")
if uploading.exists():
    raise FileExistsError(f"Có bản tải lên dở dang: {uploading}. Kiểm tra rồi xóa thủ công.")
shutil.copy2(local_zip, uploading)
if uploading.stat().st_size != local_zip.stat().st_size:
    raise IOError("Kích thước bản sao lên Drive không khớp")
os.replace(uploading, OUTPUT_ZIP)

print("✅ Dataset 3 lớp sạch:", OUTPUT_ZIP)
print("Dung lượng:", round(OUTPUT_ZIP.stat().st_size / 1024**3, 2), "GiB")
print("Sử dụng ZIP này trong notebook train tiếp theo.")